In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board  # 3x3 tuple of tuples
        self.parent = parent
        self.move = move
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

    def __eq__(self, other):
        return self.board == other.board if isinstance(other, PuzzleState) else self.board == other

    def __hash__(self):
        return hash(self.board)

    def find_blank(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c


def calculate_h2_manhattan(current_board, goal_board):
    """Calculates sum of Manhattan distances for all non-zero tiles."""
    total_distance = 0
    goal_positions = {}

    for r in range(3):
        for c in range(3):
            goal_positions[goal_board[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            tile = current_board[r][c]
            if tile != 0:
                target_r, target_c = goal_positions[tile]
                total_distance += abs(r - target_r) + abs(c - target_c)

    return total_distance


def get_valid_moves(state):
    neighbors = []
    r, c = state.find_blank()
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for dr, dc, move_name in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_board = [list(row) for row in state.board]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
            neighbors.append(PuzzleState(tuple(tuple(row) for row in new_board), move=move_name))

    return neighbors


def reconstruct_path(node):
    path = []
    curr = node
    while curr:
        path.append(curr)
        curr = curr.parent
    return path[::-1]


def A_Star_Manhattan(start_board, goal_board):
    start = PuzzleState(start_board)
    start.g = 0
    start.h = calculate_h2_manhattan(start.board, goal_board)
    start.f = start.g + start.h

    open_list = []
    heapq.heappush(open_list, start)

    open_dict = {start.board: start.g}
    closed_list = set()

    while open_list:
        current = heapq.heappop(open_list)

        if current.board == goal_board:
            return reconstruct_path(current)

        closed_list.add(current.board)

        for neighbor in get_valid_moves(current):
            if neighbor.board in closed_list:
                continue

            neighbor.g = current.g + 1
            neighbor.h = calculate_h2_manhattan(neighbor.board, goal_board)
            neighbor.f = neighbor.g + neighbor.h
            neighbor.parent = current

            if neighbor.board not in open_dict or neighbor.g < open_dict[neighbor.board]:
                open_dict[neighbor.board] = neighbor.g
                heapq.heappush(open_list, neighbor)

    return None


def print_board(board):
    for row in board:
        print(" ".join(str(val) if val != 0 else "_" for val in row))
    print()


if __name__ == "__main__":
    start_state = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    goal_state = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

    path = A_Star_Manhattan(start_state, goal_state)

    if path:
        print(f"Solved using Manhattan Distance in {len(path) - 1} steps:\n")
        for step, node in enumerate(path):
            if step == 0:
                print("Initial State:")
            else:
                print(f"Step {step}: Move Blank {node.move}")
            print_board(node.board)
    else:
        print("No solution found.")